# Complexity of calculating vanilla IMV

This notebook generates **Table A1** for the paper. It repeats the simulation in
Box 1: one Gaussian predictor, a binary logistic outcome, a 70/30 train/test split,
logistic regression, and a baseline equal to the training prevalence. Only the
complete public call `vanilla_imv(p_baseline, p_enhanced, y_test)` is timed.
Simulation, splitting, fitting, and prediction happen before timing.

Run all cells from this directory or the repository root. Use the repository's
Python environment (install from the repository root with
`python -m pip install -r requirements.txt`). The notebook
saves the summary, individual timing batches, environment metadata, and a LaTeX
table under `output/tables/`. If the local Overleaf manuscript is found, it also
updates Table A1 directly in the consolidated `main.tex`.
`IMV_MANUSCRIPT_DIR` can specify a different manuscript directory.

In [1]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from timeit import Timer
import hashlib
import inspect
import json
import os
import platform
import shutil
import subprocess

import numpy as np
import pandas as pd
from imvpy import vanilla_imv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src/simulations').is_dir()
    and (path / 'requirements.txt').is_file()
)
OUTPUT_DIR = REPO_ROOT / 'output/tables'
TOTAL_SIZES = [1_000, 10_000, 100_000, 1_000_000]
SEED = 42
REPEATS = 9

## Prepare the same predictions as Box 1

The first dataset is exactly the 1,000-observation example in Box 1. Each larger
dataset uses the same generating process and seed. The baseline and fitted model
use training observations only; all IMV calculations use the test observations.

In [2]:
def simulate_predictions(n):
    rng = np.random.default_rng(SEED)
    x = rng.normal(size=(n, 1))
    p = 1 / (1 + np.exp(-(2 * x[:, 0] - 1)))
    y = rng.binomial(1, p)
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.3, random_state=SEED
    )
    model = LogisticRegression().fit(x_train, y_train)
    p_enhanced = model.predict_proba(x_test)[:, 1]
    p_baseline = y_train.mean()
    return p_baseline, p_enhanced, y_test

## Time the complete library call

For each size, evaluate IMV once to warm up and check the result. Then
`Timer.autorange()` chooses a batch size whose calibration run lasts at least
0.2 seconds. Repeat that many calls in each of nine timing batches, using the
same inputs. Divide each batch duration by its number of calls. Report the
median and interquartile range (75th minus 25th percentile) of those nine
per-call times, and the median time per test observation.

The timer uses `perf_counter` and disables cyclic garbage collection during each
batch (the standard `timeit` behaviour). Runs are sequential, with numerical
library threads limited to one. Input validation, scalar-baseline broadcasting,
likelihood calculations, and root finding are all inside the timer. These are
repeated timings on fixed data, not independent simulation replications or
confidence intervals for IMV. Runtime depends on the machine and software.

In [3]:
rows = []
batch_rows = []
with threadpool_limits(limits=1):
    for n in TOTAL_SIZES:
        p_baseline, p_enhanced, y_test = simulate_predictions(n)
        imv = vanilla_imv(p_baseline, p_enhanced, y_test)
        if not np.isfinite(imv):
            raise ValueError(f'Undefined IMV for sample size {n}.')

        timer = Timer(lambda: vanilla_imv(p_baseline, p_enhanced, y_test))
        calls, calibration_seconds = timer.autorange()
        elapsed = np.array(timer.repeat(repeat=REPEATS, number=calls))
        per_call_ms = 1000 * elapsed / calls
        q25, median, q75 = np.quantile(per_call_ms, [0.25, 0.5, 0.75])
        rows.append({
            'n_total': n,
            'n_test': len(y_test),
            'imv': imv,
            'calls_per_batch': calls,
            'median_ms': median,
            'q25_ms': q25,
            'q75_ms': q75,
            'iqr_ms': q75 - q25,
            'us_per_observation': 1000 * median / len(y_test),
        })
        for repeat, seconds in enumerate(elapsed, start=1):
            batch_rows.append({
                'n_total': n,
                'n_test': len(y_test),
                'repeat': repeat,
                'calls': calls,
                'elapsed_seconds': seconds,
                'per_call_ms': 1000 * seconds / calls,
                'calibration_seconds': calibration_seconds,
            })
        print(f'n_test={len(y_test):>7,}: IMV={imv:.3f}, '
              f'median={median:.3f} ms, IQR={q75 - q25:.3f} ms')

summary = pd.DataFrame(rows)
batches = pd.DataFrame(batch_rows)
assert len(batches) == len(TOTAL_SIZES) * REPEATS
assert round(summary.iloc[0].imv, 3) == 0.282
assert np.isfinite(summary.to_numpy()).all()
assert summary.median_ms.gt(0).all()
summary

n_test=    300: IMV=0.282, median=0.078 ms, IQR=0.002 ms


n_test=  3,000: IMV=0.279, median=0.124 ms, IQR=0.000 ms


n_test= 30,000: IMV=0.303, median=0.719 ms, IQR=0.010 ms


n_test=300,000: IMV=0.303, median=6.987 ms, IQR=0.319 ms


,n_total,n_test,imv,calls_per_batch,median_ms,q25_ms,q75_ms,iqr_ms,us_per_observation
0,1000,300,0.281593,5000,0.078049,0.077499,0.079740,0.002241,0.260163
1,10000,3000,0.278940,2000,0.124196,0.124038,0.124514,0.000476,0.041399
2,100000,30000,0.302529,500,0.719060,0.716438,0.726775,0.010337,0.023969
3,1000000,300000,0.302510,50,6.986959,6.940881,7.260277,0.319397,0.023290


## Interpretation

For $n$ test observations, `vanilla_imv` performs input checks and two
vectorised likelihood reductions, followed by two scalar Brent root searches.
Writing their iteration counts as $k_0$ and $k_1$, the time cost is
$O(n+k_0+k_1)$, or $O(n)$ in sample size at fixed numerical precision. The current
implementation uses $O(n)$ auxiliary memory, including a broadcast baseline
array and temporary likelihood arrays. Small samples can be dominated by the
fixed cost of calling the function and finding the two roots.

These measurements quantify the additional cost of evaluating IMV **after
predictions are available**. They do not measure model training, cross-validation,
or total time per training epoch. The asymptotic argument follows the library's
operations; a finite timing experiment illustrates rather than proves it.

In [4]:
cpu = platform.processor() or platform.machine()
cpuinfo = Path('/proc/cpuinfo')
if cpuinfo.is_file():
    cpu = next((line.split(':', 1)[1].strip()
                for line in cpuinfo.read_text().splitlines()
                if line.startswith('model name')), cpu)
core_path = Path(inspect.getsourcefile(vanilla_imv))
metadata = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'cpu': cpu,
    'packages': {name: version(name) for name in
                 ('imvpy', 'numpy', 'scipy', 'scikit-learn', 'pandas')},
    'imvpy_core_sha256': hashlib.sha256(core_path.read_bytes()).hexdigest(),
    'inverse_method': inspect.signature(vanilla_imv).parameters['method'].default,
    'seed': SEED,
    'total_sizes': TOTAL_SIZES,
    'test_fraction': 0.3,
    'repeats': REPEATS,
    'calibration_target_seconds': 0.2,
    'numerical_library_threads': 1,
    'timed_call': 'vanilla_imv(p_baseline, p_enhanced, y_test)',
    'training_and_prediction_timed': False,
    'cyclic_gc_during_timing': False,
}
print(json.dumps(metadata, indent=2))

{
  "timestamp_utc": "2026-09-08T09:14:19.501685+00:00",
  "python": "3.12.7",
  "platform": "Linux-7.0.0-30-generic-x86_64-with-glibc2.43",
  "cpu": "Intel(R) Core(TM) i9-14900KF",
  "packages": {
    "imvpy": "1.2.0",
    "numpy": "1.26.4",
    "scipy": "1.16.2",
    "scikit-learn": "1.7.2",
    "pandas": "2.3.3"
  },
  "imvpy_core_sha256": "29b28b40fd8b9830cc1c4563791d037af3b56342336c5d65a0cbd0aa027f8efe",
  "inverse_method": "brentq",
  "seed": 42,
  "total_sizes": [
    1000,
    10000,
    100000,
    1000000
  ],
  "test_fraction": 0.3,
  "repeats": 9,
  "calibration_target_seconds": 0.2,
  "numerical_library_threads": 1,
  "timed_call": "vanilla_imv(p_baseline, p_enhanced, y_test)",
  "training_and_prediction_timed": false,
  "cyclic_gc_during_timing": false
}


## Export Table A1

The CSV and LaTeX table are built from the same `summary` object. The raw timing
batches and metadata make the reported summaries inspectable. Each run replaces
only this benchmark's four output files. The manuscript copy contains the full
table environment, caption, and existing `tab:time` label.

In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / 'complexity.csv', index=False)
batches.to_csv(OUTPUT_DIR / 'complexity_timings.csv', index=False)
(OUTPUT_DIR / 'complexity_metadata.json').write_text(
    json.dumps(metadata, indent=2) + '\n', encoding='utf-8'
)

table_rows = [
    f"{int(row.n_test):,} & {row.imv:.3f} & {row.median_ms:.3f} & "
    f"{row.iqr_ms:.3f} & {row.us_per_observation:.3f} " + r'\\'
    for row in summary.itertuples(index=False)
]
latex = '\n'.join([
    '% Generated by src/simulations/complexity.ipynb; do not edit by hand.',
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Runtime of \texttt{vanilla\_imv} using the simulation '
    r'in Box~\ref{cb:vanilla-imv}. Median and interquartile range (IQR) are '
    f'computed across {REPEATS} batches of repeated calls; '
    r'$n$ is the number of test observations. Fitting and prediction are excluded.}',
    r'\label{tab:time}',
    r'\small',
    r'\setlength{\tabcolsep}{4pt}',
    r'\begin{tabular}{rrrrr}',
    r'\toprule',
    r'$n$ & IMV & \shortstack{Median\\(ms)} & '
    r'\shortstack{IQR\\(ms)} & \shortstack{Time/obs.\\($\mu$s)} \\',
    r'\midrule',
    *table_rows,
    r'\bottomrule',
    r'\end{tabular}',
    r'\par\smallskip',
    r'\raggedright\footnotesize',
    r'\texttt{imvpy} ' + metadata['packages']['imvpy']
    + '; Python ' + metadata['python'] + '; ' + metadata['cpu']
    + '; one numerical-library thread.',
    r'\end{table}',
    '',
])
table_path = OUTPUT_DIR / 'complexity.tex'
table_path.write_text(latex, encoding='utf-8')
print(f'Saved benchmark tables and metadata to {OUTPUT_DIR}')

manuscript_override = os.environ.get('IMV_MANUSCRIPT_DIR')
if manuscript_override:
    manuscript_dir = Path(manuscript_override).expanduser().resolve()
    if not any((manuscript_dir / name).is_file()
               for name in ('main.tex', '9_sup.tex')):
        raise FileNotFoundError(f'Manuscript not found: {manuscript_dir}')
else:
    manuscript_dir = next(
        (parent / 'Apps/Overleaf/IMV_ML_paper' for parent in REPO_ROOT.parents
         if any((parent / 'Apps/Overleaf/IMV_ML_paper' / name).is_file()
                for name in ('main.tex', '9_sup.tex'))),
        None,
    )

if manuscript_dir is None:
    print('To update Table A1 locally, set IMV_MANUSCRIPT_DIR and rerun this cell.')
else:
    sync_tool = shutil.which('overleaf-sync-now')
    if 'Overleaf' in manuscript_dir.parts and sync_tool is None:
        raise RuntimeError('Table files were exported, but overleaf-sync-now is '
                           'needed to refresh this Overleaf manuscript before copying.')
    if sync_tool and 'Overleaf' in manuscript_dir.parts:
        subprocess.run([sync_tool, 'sync', str(manuscript_dir)], check=True)
    manuscript_main = manuscript_dir / 'main.tex'
    if manuscript_main.is_file():
        main_source = manuscript_main.read_text(encoding='utf-8')
        begin_marker = '% BEGIN INPUT: tables/complexity.tex\n'
        end_marker = '% END INPUT: tables/complexity.tex\n'
        if (main_source.count(begin_marker) != 1
                or main_source.count(end_marker) != 1
                or main_source.find(begin_marker) >= main_source.find(end_marker)):
            raise ValueError('Expected one marked complexity table in main.tex.')
        before, _, remainder = main_source.partition(begin_marker)
        _, _, after = remainder.partition(end_marker)
        updated_main = before + begin_marker + latex + end_marker + after
        manuscript_main.write_text(updated_main, encoding='utf-8')
        print(f'Updated Table A1 in consolidated manuscript: {manuscript_main}')
    else:
        manuscript_table = manuscript_dir / 'tables/complexity.tex'
        manuscript_table.parent.mkdir(exist_ok=True)
        shutil.copyfile(table_path, manuscript_table)
        assert manuscript_table.read_bytes() == table_path.read_bytes()
        print(f'Updated Table A1 source: {manuscript_table}')


Saved benchmark tables and metadata to /home/jinx/CharlieRahal Dropbox/Charlie RAHAL/imv/imv_ml/new/IMV_ML/output/tables


Refreshing project 66e593b4ab8ee6c91073fbee (folder: /home/jinx/CharlieRahal Dropbox/Charlie RAHAL/Apps/Overleaf/IMV_ML_paper)
dropbox_echo (toV 5904->5905) (1.05s)
Updated Table A1 source: /home/jinx/CharlieRahal Dropbox/Charlie RAHAL/Apps/Overleaf/IMV_ML_paper/tables/complexity.tex
